# Airbnb NYC 2019 - EDA y preparación de datos

Este notebook sigue los pasos del ejercicio: carga, EDA, split train/test y guardado de datos procesados.

In [ ]:
from pathlib import Path
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
DATA_RAW_DIR = ROOT_DIR / 'data' / 'raw'
DATA_PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
URL = 'https://breathecode.herokuapp.com/asset/internal-link?id=927&path=AB_NYC_2019.csv'
RAW_PATH = DATA_RAW_DIR / 'AB_NYC_2019.csv'

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    response = requests.get(URL, timeout=120)
    response.raise_for_status()
    RAW_PATH.write_bytes(response.content)

RAW_PATH

In [ ]:
df = pd.read_csv(RAW_PATH)
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)
print('\nNulos por columna:')
print(df.isna().sum())
print('\nDuplicados:', df.duplicated().sum())
print('\nResumen precio:')
print(df['price'].describe())

In [ ]:
selected_columns = [
    'neighbourhood_group',
    'neighbourhood',
    'latitude',
    'longitude',
    'room_type',
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365',
    'price'
]

model_df = df[selected_columns].drop_duplicates().copy()
model_df = model_df[model_df['price'] > 0]

X = model_df.drop(columns=['price'])
y = model_df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = [
    'latitude', 'longitude', 'minimum_nights', 'number_of_reviews',
    'reviews_per_month', 'calculated_host_listings_count', 'availability_365'
]
categorical_features = ['neighbourhood_group', 'neighbourhood', 'room_type']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)
y_train_df = y_train.to_frame(name='price')
y_test_df = y_test.to_frame(name='price')

X_train_df.shape, X_test_df.shape

In [ ]:
X_train_df.to_csv(DATA_PROCESSED_DIR / 'X_train.csv', index=False)
X_test_df.to_csv(DATA_PROCESSED_DIR / 'X_test.csv', index=False)
y_train_df.to_csv(DATA_PROCESSED_DIR / 'y_train.csv', index=False)
y_test_df.to_csv(DATA_PROCESSED_DIR / 'y_test.csv', index=False)

print('Guardado en:', DATA_PROCESSED_DIR)